# Commodity Forwards in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Transaction types |
| 4 | Portfolio and transactions |
| 5 | Valuation |
| 6 | Instrument events |

## The instrument

A commodity forward is an agreement to exchange a commodity at a fixed **strike** on a fixed
**maturity date**. Before maturity it's worth the gap between the strike and the market. At
maturity it settles.

`delivery_type` decides how:

| `delivery_type` | At maturity |
|---|---|
| `Cash` | the difference is paid in cash, position closes |
| `Physical` | the commodity is delivered, the forward's cost basis moves onto it |

LUSID derives the settlement event from `maturity_date` and `delivery_type` -- you never post it
yourself. Participation is `Mandatory`, so the holder gets no election.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

Both contracts share a maturity so the two settlement styles can be compared side by side. `ASOF`
sits the day before that maturity, so both contracts are still live when this notebook values
them -- section 6's events window then runs well past maturity, to where settlement actually gets
forecast.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "CommodityForwardDemo"
RECIPE    = "commodity-forward-demo-recipe"
PORTFOLIO = "commodity-forward-demo-book"

START    = d(2025, 1, 15)      # trade date
MATURITY = d(2025, 12, 31)     # maturity, delivery date, event date
ASOF     = d(2025, 12, 30)     # valuation date -- the day before maturity
AFTER    = d(2026, 1, 15)      # after maturity, for the events window

UNITS        = 100
GOLD_STRIKE  = 2_050.00        # USD / troy oz, physically delivered
WTI_STRIKE   = 71.50           # USD / barrel, cash settled
HOLDING_COST = 100.00          # cost of putting the forward on
FUNDING      = 1_000_000.00    # opening cash

GOLD_MARKET  = 2_180.00
WTI_MARKET   = 78.25

print(f"Scope     : {SCOPE}")
print(f"Contract  : {START:%Y-%m-%d} -> {MATURITY:%Y-%m-%d}")
print(f"Valued at : {ASOF:%Y-%m-%d}")
print(f"Events at : {AFTER:%Y-%m-%d}")
print()
print(f"Gold  strike {GOLD_STRIKE:>8,.2f}  market {GOLD_MARKET:>8,.2f}  "
      f"gain {GOLD_MARKET - GOLD_STRIKE:>7,.2f}/oz  x {UNITS} = "
      f"{(GOLD_MARKET - GOLD_STRIKE) * UNITS:>10,.2f}")
print(f"WTI   strike {WTI_STRIKE:>8,.2f}  market {WTI_MARKET:>8,.2f}  "
      f"gain {WTI_MARKET - WTI_STRIKE:>7,.2f}/bbl x {UNITS} = "
      f"{(WTI_MARKET - WTI_STRIKE) * UNITS:>10,.2f}")

Scope     : CommodityForwardDemo
Contract  : 2025-01-15 -> 2025-12-31
Valued at : 2025-12-30
Events at : 2026-01-15

Gold  strike 2,050.00  market 2,180.00  gain  130.00/oz  x 100 =  13,000.00
WTI   strike    71.50  market    78.25  gain    6.75/bbl x 100 =     675.00


---
# 1. Instrument creation

## 1a. The deliverable commodity

A physically-settled forward has to say what gets delivered. That underlying is a mastered
`SimpleInstrument` with `asset_class="Commodities"` -- it has to exist as its own instrument
because after delivery you hold a position in it directly.

A cash-settled forward has no underlying at all.

In [3]:
gold = m.SimpleInstrument(
    instrument_type="SimpleInstrument",
    dom_ccy="USD",
    asset_class="Commodities",
    simple_instrument_type="Commodity",
    maturity_date=d(2030, 12, 31))

GOLD_LUID = upsert("gold", "Gold (physical commodity, troy oz)", "CF-DEMO-GOLD", gold)
print(f"Gold : {GOLD_LUID}")

Gold : LUID_00003DCX


## 1b. Cash-settled forward

Start, maturity, currency, strike, `delivery_type` -- that's all a cash-settled forward needs.

In [4]:
wti_cash = m.CommodityForward(
    instrument_type="CommodityForward",
    start_date=START,
    maturity_date=MATURITY,
    dom_ccy="USD",
    strike=WTI_STRIKE,
    delivery_type="Cash")

WTI_LUID = upsert("wti", f"WTI Crude Forward Dec-25 @ {WTI_STRIKE} (cash settled)",
                  "CF-DEMO-WTI-CASH", wti_cash)
print(f"WTI cash-settled forward : {WTI_LUID}")

WTI cash-settled forward : LUID_00003DCY


## 1c. Physical forward

A physically-settled forward needs the same fields as the cash-settled one, plus `underlying`
pointing at the mastered commodity from 1a -- that's what makes physical delivery possible.

In [5]:
gold_physical = m.CommodityForward(
    instrument_type="CommodityForward",
    start_date=START,
    maturity_date=MATURITY,
    dom_ccy="USD",
    strike=GOLD_STRIKE,
    delivery_type="Physical",
    underlying=mastered(GOLD_LUID))

GOLDFWD_LUID = upsert("goldfwd", f"Gold Forward Dec-25 @ {GOLD_STRIKE} (physically delivered)",
                      "CF-DEMO-GOLD-PHYS", gold_physical)
print(f"Gold physical forward : {GOLDFWD_LUID}\n")

display(pd.DataFrame([
    {"contract": "WTI Crude Dec-25", "luid": WTI_LUID, "deliveryType": "Cash",
     "strike": WTI_STRIKE, "delivers": "-"},
    {"contract": "Gold Dec-25", "luid": GOLDFWD_LUID, "deliveryType": "Physical",
     "strike": GOLD_STRIKE, "delivers": GOLD_LUID},
]))

Gold physical forward : LUID_00003DCZ



,contract,luid,deliveryType,strike,delivers
0,WTI Crude Dec-25,LUID_00003DCY,Cash,71.50,-
1,Gold Dec-25,LUID_00003DCZ,Physical,"2,050.00",LUID_00003DCX


---
# 2. Recipe

Used for both valuation and event forecasting. Quotes are keyed on each instrument's own LUID.

In [6]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Commodity forward demo",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: CommodityForwardDemo/commodity-forward-demo-recipe


---
# 3. Transaction types

A transaction type maps a label to a holdings movement. A settlement template emits transactions
of a given type, so those types have to exist first, or the template has nothing to build.

Physical settlement emits two: a close of the forward and a delivery of the commodity. Cash
settlement emits one. `direction` is `-1` to remove units, `+1` to add them.

In [7]:
TXN_TYPES = [
    ("CommodityForwardPhysicalClose",    "Close forward on physical delivery", -1),
    ("CommodityForwardPhysicalDelivery", "Deliver the commodity",              +1),
    ("CommodityForwardCashSettlement",   "Cash settle the forward",            -1),
]

txn_config_api = api(lusid.TransactionConfigurationApi)

for txn_type, description, direction in TXN_TYPES:
    txn_config_api.set_transaction_type(
        source="default", type=txn_type, scope="default",
        transaction_type_request=m.TransactionTypeRequest(
            aliases=[m.TransactionTypeAlias(
                type=txn_type, description=description,
                transaction_class="Basic", transaction_roles="AllRoles", is_default=False)],
            movements=[m.TransactionTypeMovement(
                movement_types="StockMovement", side="Side1", direction=direction)]))
    print(f"{txn_type:<36} StockMovement Side1 {direction:+d}")

CommodityForwardPhysicalClose        StockMovement Side1 -1


CommodityForwardPhysicalDelivery     StockMovement Side1 +1
CommodityForwardCashSettlement       StockMovement Side1 -1


---
# 4. Portfolio and transactions

Fund the book, then buy both forwards. `total_consideration` is what the position cost, not the
notional -- no principal changes hands when a forward is struck, so a holding cost of 100 buys a
contract with a notional of 205,000. That cost becomes the basis that moves onto the delivered
commodity at physical settlement.

In [8]:
recreate_portfolio(PORTFOLIO, "Commodity Forward Demo Book", "USD", d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="FUNDING",
        type="FundsIn",
        instrument_identifiers={"Instrument/default/Currency": "USD"},
        transaction_date=START.isoformat(),
        settlement_date=START.isoformat(),
        units=FUNDING,
        transaction_price=m.TransactionPrice(price=1.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=FUNDING, currency="USD"),
        source="default")])


def buy(luid, units, consideration, txn_id):
    txn_portfolio_api.upsert_transactions(
        scope=SCOPE, code=PORTFOLIO,
        transaction_request=[m.TransactionRequest(
            transaction_id=txn_id,
            type="Buy",
            instrument_identifiers={"Instrument/default/LusidInstrumentId": luid},
            transaction_date=START.isoformat(),
            settlement_date=START.isoformat(),
            units=units,
            transaction_price=m.TransactionPrice(price=0.0, type="Price"),
            total_consideration=m.CurrencyAndAmount(amount=consideration, currency="USD"),
            source="default")])


buy(WTI_LUID,     UNITS, HOLDING_COST, "BUY-WTI-CASH")
buy(GOLDFWD_LUID, UNITS, HOLDING_COST, "BUY-GOLD-PHYS")

display(transactions(PORTFOLIO, START, ASOF))

Recreated CommodityForwardDemo/commodity-forward-demo-book


,date,type,luid,units,consideration
0,2025-01-15,Buy,LUID_00003DCY,100.00,100.00
1,2025-01-15,Buy,LUID_00003DCZ,100.00,100.00
2,2025-01-15,FundsIn,CCY_USD,"1,000,000.00","1,000,000.00"


---
# 5. Valuation

Each forward's own quote is its gain per unit over the strike, so `Valuation/PV` is that gain
across the position.

In [9]:
marks = {
    WTI_LUID:     WTI_MARKET - WTI_STRIKE,
    GOLDFWD_LUID: GOLD_MARKET - GOLD_STRIKE,
    GOLD_LUID:    GOLD_MARKET,
}
for luid, price in marks.items():
    upsert_price(luid, price, ASOF, "USD")
print(f"Quotes loaded for {len(marks)} instruments\n")

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/PV",            "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, "USD")
display(result)

names = {"WTI Crude Forward Dec-25 @ 71.5 (cash settled)": WTI_MARKET - WTI_STRIKE,
         f"Gold Forward Dec-25 @ {GOLD_STRIKE} (physically delivered)": GOLD_MARKET - GOLD_STRIKE}
for name, gain in names.items():
    pv = result.loc[result["Instrument/default/Name"] == name, "Sum(Valuation/PV)"].iloc[0]
    print(f"LUSID PV {pv:>10,.2f}  vs  gain x units {gain * UNITS:>10,.2f}   {name}")

Quotes loaded for 3 instruments



,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/PV)
0,USD,"999,800.00","999,800.00"
1,WTI Crude Forward Dec-25 @ 71.5 (cash settled),100.00,675.00
2,Gold Forward Dec-25 @ 2050.0 (physically deliv...,100.00,"13,000.00"


LUSID PV     675.00  vs  gain x units     675.00   WTI Crude Forward Dec-25 @ 71.5 (cash settled)
LUSID PV  13,000.00  vs  gain x units  13,000.00   Gold Forward Dec-25 @ 2050.0 (physically delivered)


---
# 6. Instrument events

Nothing above posts an event. LUSID reads `maturity_date` and `delivery_type` off each instrument
and derives what happens:

| `delivery_type` | Event |
|---|---|
| `Cash` | `CommodityForwardCashSettlementEvent` |
| `Physical` | `CommodityForwardPhysicalSettlementEvent` |

## 6a. Applicable events

`query_applicable_instrument_events` forecasts every event the portfolio's holdings imply over a
window. Expect four: a settlement event and a `MaturityEvent` per contract.

In [10]:
events_api = api(lusid.InstrumentEventsApi)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=START.isoformat(),
        window_end=AFTER.isoformat(),
        effective_at=AFTER.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

instrument_names = {WTI_LUID: "WTI (cash)", GOLDFWD_LUID: "Gold (physical)",
                    GOLD_LUID: "Gold commodity"}

print(f"{len(applicable)} applicable event(s):\n")
display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "instrument": instrument_names.get(ev.lusid_instrument_id, ev.lusid_instrument_id),
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

4 applicable event(s):



,event type,instrument,eligible balance,status
0,CommodityForwardCashSettlementEvent,WTI (cash),100.00,Active
1,CommodityForwardPhysicalSettlementEvent,Gold (physical),100.00,Active
2,MaturityEvent,WTI (cash),0.00,Active
3,MaturityEvent,Gold (physical),0.00,Active


## 6b. The transactions each event would book

Every applicable event carries the transactions its template would produce, using the transaction
types registered in section 3.

In [11]:
rows = []
for ev in applicable:
    for txn in (ev.transactions or []):
        rows.append({
            "event": ev.instrument_event_type.replace("CommodityForward", "CF."),
            "on": instrument_names.get(ev.lusid_instrument_id, ev.lusid_instrument_id),
            "txn type": getattr(txn, "type", None),
            "units": getattr(txn, "units", None),
            "price": getattr(getattr(txn, "transaction_price", None), "price", None),
            "consideration": getattr(getattr(txn, "total_consideration", None), "amount", None),
        })

if rows:
    display(pd.DataFrame(rows))
    print("Physical settlement books two legs, cash settlement one.")
else:
    print("No forecast transactions. Check the transaction types in section 3.")

,event,on,txn type,units,price,consideration
0,CF.CashSettlementEvent,WTI (cash),CommodityForwardCashSettlement,100.00,6.75,675.00
1,CF.PhysicalSettlementEvent,Gold (physical),CommodityForwardPhysicalClose,100.00,0.00,0.00
2,CF.PhysicalSettlementEvent,Gold (physical),CommodityForwardPhysicalDelivery,100.00,"2,050.00","205,100.00"


Physical settlement books two legs, cash settlement one.


## 6c. Template specifications

Each event type publishes the fields its template can use, which instruments it applies to, and
whether the holder gets a choice. `supportedParticipationTypes` is `Mandatory` for both.

In [12]:
event_types_api = api(lusid.InstrumentEventTypesApi)

for event_type in ("CommodityForwardCashSettlementEvent",
                   "CommodityForwardPhysicalSettlementEvent"):
    sd = json.loads(event_types_api.get_transaction_template_specification(
        instrument_event_type=event_type).to_json())
    fields = [f.get("fieldName") for f in (sd.get("supportedTemplateFields") or [])]
    print(event_type)
    print(f"  applies to    : {sd.get('supportedInstrumentTypes')}")
    print(f"  participation : {sd.get('supportedParticipationTypes')}")
    print(f"  eligibility   : {sd.get('eligibilityCalculation')}")
    print(f"  fields ({len(fields)}) : {', '.join(fields)}")
    print()

CommodityForwardCashSettlementEvent
  applies to    : ['CommodityForward']
  participation : ['Mandatory']
  eligibility   : {'entitlementDate': 'maturityDate', 'eligibleUnits': 'SettledUnits', 'dateModifiableByInstruction': False}
  fields (19) : holdingId, rootPortfolioHoldingId, holdingCurrency, holdingCost, holdingNotionalCost, holdingVariationMargin, portfolioCurrency, holdingCostPortfolioCurrency, holdingVariationMarginPortfolioCurrency, holdingCostToPortfolioFxRate, instrumentEventId, instrument, completeness, eligibleBalance, maturityDate, cashFlowPerUnit, cashFlowAmount, domCcy, strike



CommodityForwardPhysicalSettlementEvent
  applies to    : ['CommodityForward']
  participation : ['Mandatory']
  eligibility   : {'entitlementDate': 'maturityDate', 'eligibleUnits': 'SettledUnits', 'dateModifiableByInstruction': False}
  fields (20) : holdingId, rootPortfolioHoldingId, holdingCurrency, holdingCost, holdingNotionalCost, holdingVariationMargin, portfolioCurrency, holdingCostPortfolioCurrency, holdingVariationMarginPortfolioCurrency, holdingCostToPortfolioFxRate, instrumentEventId, instrument, completeness, eligibleBalance, maturityDate, strike, underlyingInstrument, underlyingInstrumentCurrency, deliveredUnits, underlyingTotalConsideration



## 6d. Applying events

Sections 6a and 6b are forecasts. To have settlement write real transactions into the book, the
portfolio also needs a corporate action source, created with the portfolio -- the same channel
applied events arrive on generally, named after its original use for dividends and splits.

| To get | Setup required |
|---|---|
| which events apply | `instrumentEventConfiguration` on the portfolio |
| what they would book | plus transaction types (section 3) |
| them booked into the book | plus a corporate action source |

---
# Summary

1. `delivery_type` selects the settlement event, and therefore what happens at maturity.
2. A physical forward references a mastered commodity through `underlying` -- that's what makes
   physical delivery possible, and it's absent entirely from a cash-settled forward.
3. Settlement is derived from the instrument and is `Mandatory` -- you never post it yourself.
4. A transaction type has to exist before a settlement template has anything to build.
5. `instrumentEventConfiguration` is create-time only on the portfolio. Passing `recipe=RECIPE`
   into `recreate_portfolio()` in section 4 took care of that.

In [13]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Instruments: {GOLD_LUID} gold, {WTI_LUID} WTI cash, {GOLDFWD_LUID} gold physical")

Scope      : CommodityForwardDemo
Portfolio  : CommodityForwardDemo/commodity-forward-demo-book
Recipe     : CommodityForwardDemo/commodity-forward-demo-recipe
Instruments: LUID_00003DCX gold, LUID_00003DCY WTI cash, LUID_00003DCZ gold physical
